# University Waste Management & Agricultural Field Monitoring System
### HSV Green-Masking + YOLOv8 — Colab GPU Pipeline

This notebook runs the full pipeline end-to-end on a Colab GPU runtime:

1. Environment setup (GPU check, Drive mount, dependencies)
2. Dataset download (Kaggle) + sanity inspection
3. Dataset assembly: single-class merge, train/val/test split
4. HSV green-masking preprocessing (visual pipeline check)
5. Baseline (unmasked) vs HSV-masked training — the core ablation
6. Metrics comparison (mAP50, mAP50-95, precision, recall)
7. Inference on original RGB images with green-ratio post-filter
8. Persist everything to Google Drive

**Before running:** `Runtime > Change runtime type > T4 GPU` (or better).

## 1. Environment setup

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/university-waste-management'
import os
os.makedirs(PROJECT_ROOT, exist_ok=True)
print('Project root:', PROJECT_ROOT)

Clone the project repo (containing `src/`) into the Colab runtime. Replace `REPO_URL`
if you've pushed this project to GitHub — otherwise upload the `src/` folder manually via the
Colab file browser into `/content/University-Waste-Management/`.

In [1]:
REPO_URL = 'https://github.com/ahmedtayel714/University-Waste-Management.git'
CODE_DIR = '/content/University-Waste-Management'

if REPO_URL:
    !git clone -q {REPO_URL} {CODE_DIR}
else:
    os.makedirs(CODE_DIR, exist_ok=True)
    print('REPO_URL not set — upload/sync src/ into', CODE_DIR, 'manually before continuing.')

import sys
sys.path.insert(0, CODE_DIR)

NameError: name 'os' is not defined

### Resuming a previous session?

If you already trained baseline/masked/waste/leaf models in an earlier
session, run the cell below to recreate the path variables those training
cells would have set — **without re-running training**. Skip it on a
genuinely first run (the variables get set naturally as you go through the
notebook the first time).

In [ ]:
RESUMING_PREVIOUS_SESSION = False  # flip to True if models already exist on Drive

if RESUMING_PREVIOUS_SESSION:
    from pathlib import Path
    from src.preprocessing.hsv_mask import MaskConfig

    RUNS_DIR = f'{PROJECT_ROOT}/runs'
    RAW_DATA_DIR = f'{PROJECT_ROOT}/data/raw'
    BASELINE_DIR = Path(PROJECT_ROOT) / 'data' / 'baseline'
    MASKED_DIR = Path(PROJECT_ROOT) / 'data' / 'masked'
    WASTE_RAW_DIR = f'{PROJECT_ROOT}/data/waste_raw'

    baseline_run_dir = f'{RUNS_DIR}/baseline'
    masked_run_dir = f'{RUNS_DIR}/masked'
    masked_weights = f'{RUNS_DIR}/masked/weights/best.pt'
    waste_weights = f'{RUNS_DIR}/waste/weights/best.pt'
    leaf_weights = f'{RUNS_DIR}/leaf/weights/best.pt'
    leaf_v2_weights = f'{RUNS_DIR}/leaf_v2/weights/best.pt'

    mask_config = MaskConfig()

    for name, path in [('masked_weights', masked_weights), ('waste_weights', waste_weights), ('leaf_weights', leaf_weights), ('leaf_v2_weights', leaf_v2_weights)]:
        exists = Path(path).exists()
        print(f'{name}: {"found" if exists else "NOT FOUND"} at {path}')
else:
    print('Fresh run — variables will be set naturally as you go.')

In [ ]:
!pip install -q ultralytics opencv-python-headless kaggle pyyaml tqdm seaborn

### All project imports (run this once per session)

Every `src/` function used anywhere in this notebook, imported up front —
so skipping a training/generation cell you don't need (because the model
or dataset already exists on Drive) never leaves you with a `NameError`
for something that cell happened to import as a side effect. Individual
cells further down still show their own imports too, as documentation of
what each step actually needs — re-importing an already-imported name is
a no-op, so there's no harm running both.

In [ ]:
from pathlib import Path
import cv2
import json

from src.preprocessing.hsv_mask import MaskConfig, visualize_pipeline, green_mask
from src.preprocessing.dataset_prep import (
    discover_pairs, inspect_class_distribution, find_class_names,
    split_dataset, write_data_yaml, build_masked_variant,
)
from src.training.train import TrainConfig, train, validate
from src.evaluation.metrics import (
    load_results_csv, summarize_final_metrics, plot_loss_curves, plot_map_curves, compare_runs,
)
from src.evaluation.report import plot_pipeline_grid
from src.inference.predict import predict_image, predict_and_save, draw_detections, Detection
from src.inference.combined_predict import predict_combined, predict_combined_and_save
from src.analysis.leaf_counter import count_leaves, count_leaves_in_regions

# Track B (only needed once you reach Part 2 — harmless to import now too)
from src.synthetic.cutout_extractor import extract_cutouts_from_yolo_seg_dataset, cutout_from_bbox
from src.synthetic.background_harvester import harvest_from_green_dataset, harvest_from_boxed_dataset
from src.synthetic.compositor import compose_scene
from src.synthetic.generate_dataset import generate_synthetic_dataset
from src.inference.track import predict_image_response, track_video_to_json, track_video, TrackSmoother

# Part 3 — roadmap improvements
from src.synthetic.pipeline import build_leaf_dataset
from src.inference.error_logger import LowConfidenceLogger
from src.evaluation.field_validation import run_field_validation

print('All src/ modules imported.')

## 2. Dataset download (Kaggle)

Upload your `kaggle.json` API token when prompted (Kaggle account → Settings → Create New API Token).

In [ ]:
from google.colab import files
import os

kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)

if not os.path.exists(f'{kaggle_dir}/kaggle.json'):
    uploaded = files.upload()  # select kaggle.json
    for fname in uploaded:
        os.rename(fname, f'{kaggle_dir}/kaggle.json')
    os.chmod(f'{kaggle_dir}/kaggle.json', 0o600)
print('Kaggle credentials ready.')

In [ ]:
RAW_DATA_DIR = f'{PROJECT_ROOT}/data/raw'
os.makedirs(RAW_DATA_DIR, exist_ok=True)

!kaggle datasets download -d ravirajsinh45/crop-and-weed-detection-data-with-bounding-boxes -p {RAW_DATA_DIR} --unzip
!find {RAW_DATA_DIR} -maxdepth 3 | head -30

## 3. Dataset assembly

Discover image/label pairs, **inspect the class distribution before trusting the 0/1 class-id convention**, merge crop+weed into a single `green_vegetation` class, and split train/val/test.

In [ ]:
from pathlib import Path
from src.preprocessing.dataset_prep import (
    discover_pairs, inspect_class_distribution, split_dataset, write_data_yaml
)

pairs = discover_pairs(Path(RAW_DATA_DIR))
print(f'Found {len(pairs)} image/label pairs')
print('Class distribution (verify before merging!):', inspect_class_distribution(pairs))

In [ ]:
BASELINE_DIR = Path(PROJECT_ROOT) / 'data' / 'baseline'
split_dataset(pairs, BASELINE_DIR, train=0.7, val=0.2, test=0.1, seed=42, merge_to_single_class=True)

baseline_yaml = write_data_yaml(BASELINE_DIR / 'data.yaml', BASELINE_DIR, names=['green_vegetation'])
print('Baseline data.yaml:', baseline_yaml)

## 4. HSV green-masking — visual sanity check

Inspect the mask and both masking strategies (hard black-out vs soft desaturation) on a few sample images before committing to a full-dataset pass.

In [ ]:
import cv2
import matplotlib.pyplot as plt
from src.preprocessing.hsv_mask import MaskConfig, visualize_pipeline

sample_paths = list((BASELINE_DIR / 'images' / 'train').iterdir())[:3]
mask_config = MaskConfig()  # tune DEFAULT_LOWER_GREEN/UPPER_GREEN in hsv_mask.py if needed

fig, axes = plt.subplots(len(sample_paths), 4, figsize=(16, 4 * len(sample_paths)))
for row, img_path in enumerate(sample_paths):
    image = cv2.imread(str(img_path))
    mask, hard, soft = visualize_pipeline(image, mask_config)
    for col, (title, img) in enumerate([
        ('original', image), ('mask', mask), ('hard-masked', hard), ('soft-masked', soft)
    ]):
        ax = axes[row, col] if len(sample_paths) > 1 else axes[col]
        cmap = 'gray' if img.ndim == 2 else None
        disp = img if img.ndim == 2 else cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(disp, cmap=cmap)
        ax.set_title(title)
        ax.axis('off')
fig.tight_layout()
plt.show()

**If the mask misses vegetation or catches soil/straw**, tune `DEFAULT_LOWER_GREEN` /
`DEFAULT_UPPER_GREEN` in `src/preprocessing/hsv_mask.py` (or pass a custom `MaskConfig` above)
and re-run this cell before generating the full masked dataset.

In [ ]:
from src.preprocessing.dataset_prep import build_masked_variant

MASKED_DIR = Path(PROJECT_ROOT) / 'data' / 'masked'
build_masked_variant(BASELINE_DIR, MASKED_DIR, mode='soft', config=mask_config)
masked_yaml = write_data_yaml(MASKED_DIR / 'data.yaml', MASKED_DIR, names=['green_vegetation'])
print('Masked data.yaml:', masked_yaml)

## 5. Train — baseline vs HSV-masked ablation

Same architecture, same hyperparameters, only the preprocessing differs. This is the core evidence for the project's thesis.

In [ ]:
from src.training.train import TrainConfig, train

RUNS_DIR = f'{PROJECT_ROOT}/runs'

baseline_cfg = TrainConfig(
    data_yaml=str(baseline_yaml),
    model='yolov8s.pt',
    epochs=100,
    imgsz=512,
    batch=16,
    project=RUNS_DIR,
    name='baseline',
)
baseline_model, baseline_results = train(baseline_cfg)

In [ ]:
masked_cfg = TrainConfig(
    data_yaml=str(masked_yaml),
    model='yolov8s.pt',
    epochs=100,
    imgsz=512,
    batch=16,
    project=RUNS_DIR,
    name='masked',
)
masked_model, masked_results = train(masked_cfg)

## 6. Compare metrics

In [ ]:
from src.evaluation.metrics import load_results_csv, plot_loss_curves, plot_map_curves, compare_runs

baseline_run_dir = f'{RUNS_DIR}/baseline'
masked_run_dir = f'{RUNS_DIR}/masked'

baseline_df = load_results_csv(baseline_run_dir)
masked_df = load_results_csv(masked_run_dir)

plot_loss_curves(baseline_df, f'{PROJECT_ROOT}/reports/baseline_loss.png', 'Baseline — Loss')
plot_loss_curves(masked_df, f'{PROJECT_ROOT}/reports/masked_loss.png', 'HSV-Masked — Loss')
plot_map_curves(baseline_df, f'{PROJECT_ROOT}/reports/baseline_map.png', 'Baseline — mAP')
plot_map_curves(masked_df, f'{PROJECT_ROOT}/reports/masked_map.png', 'HSV-Masked — mAP')

summary = compare_runs(baseline_run_dir, masked_run_dir, f'{PROJECT_ROOT}/reports/comparison.png')
summary

## 7. Inference on original RGB images

The masked model still runs on **unmasked** images at inference time — masking is a training-time noise filter only. The optional green-ratio post-filter drops boxes that don't actually contain green pixels.

In [ ]:
from src.inference.predict import predict_and_save

test_images = list((BASELINE_DIR / 'images' / 'test').iterdir())[:5]
masked_weights = f'{masked_run_dir}/weights/best.pt'

for img_path in test_images:
    out_path = f'{PROJECT_ROOT}/reports/inference/{img_path.stem}_pred.jpg'
    predict_and_save(
        masked_weights, img_path, out_path,
        conf=0.25, green_ratio_threshold=0.15, mask_config=mask_config,
    )
print('Saved annotated predictions to', f'{PROJECT_ROOT}/reports/inference/')

In [ ]:
import matplotlib.pyplot as plt
import cv2
from pathlib import Path

preds = sorted(Path(f'{PROJECT_ROOT}/reports/inference').glob('*_pred.jpg'))[:5]
fig, axes = plt.subplots(1, len(preds), figsize=(4 * len(preds), 4))
for ax, p in zip(axes, preds):
    ax.imshow(cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB))
    ax.set_title(p.stem)
    ax.axis('off')
fig.tight_layout()
plt.show()

## 7b. Combined pipeline figure — before → mask → detection → leaf count

The report/competition-ready artifact: one figure per sample showing
**Original → Green Mask → Masked (training view) → Final Detection → Leaf
Count**, side by side, so the whole preprocessing + counting story is
visible at a glance. Leaf counting is a classical watershed split on the
mask (see `src/analysis/leaf_counter.py`) — an estimate, not ground truth;
it under-counts leaves that fully overlap in the 2D projection. Tune
`leaf_min_area` / `leaf_fg_ratio` below if counts look off for your images.

In [ ]:
from src.evaluation.report import plot_pipeline_grid

demo_images = list((BASELINE_DIR / 'images' / 'test').iterdir())[:5]

pipeline_fig_path = plot_pipeline_grid(
    demo_images,
    masked_weights,
    f'{PROJECT_ROOT}/reports/pipeline_demo.png',
    mask_config=mask_config,
    conf=0.25,
    green_ratio_threshold=0.15,
)
print('Saved pipeline figure to', pipeline_fig_path)

In [ ]:
import matplotlib.pyplot as plt
import cv2

fig_img = cv2.cvtColor(cv2.imread(str(pipeline_fig_path)), cv2.COLOR_BGR2RGB)
plt.figure(figsize=(16, 4 * len(demo_images)))
plt.imshow(fig_img)
plt.axis('off')
plt.show()

## 7c. Test on your own photos

Upload images straight from your device to sanity-check the trained model
on real-world shots outside the dataset's distribution (different framing,
lighting, distance) — including the leaf count. If detections come back
empty, try `conf=0.1` and `green_ratio_threshold=None` first to see raw
model confidence before the post-filters — that's a distribution-shift
finding worth reporting, not necessarily a bug.

In [ ]:
from google.colab import files
from pathlib import Path

upload_dir = Path(f'{PROJECT_ROOT}/data/custom_test')
upload_dir.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()  # pick photos from your device
custom_images = []
for fname, content in uploaded.items():
    dest = upload_dir / fname
    dest.write_bytes(content)
    custom_images.append(dest)

print(f'Uploaded {len(custom_images)} image(s) to {upload_dir}')

In [ ]:
custom_fig_path = plot_pipeline_grid(
    custom_images,
    masked_weights,
    f'{PROJECT_ROOT}/reports/custom_test_pipeline.png',
    mask_config=mask_config,
    conf=0.25,
    green_ratio_threshold=0.15,
)

import matplotlib.pyplot as plt, cv2
fig_img = cv2.cvtColor(cv2.imread(str(custom_fig_path)), cv2.COLOR_BGR2RGB)
plt.figure(figsize=(16, 4 * len(custom_images)))
plt.imshow(fig_img)
plt.axis('off')
plt.show()

## 8. Waste detection — second model

A separate, independent model for non-green waste categories (trash,
paper, plastic, soil anomalies, etc.) — deliberately **not** merged into
the vegetation model, so nothing here can change the baseline-vs-masked
ablation results you already have. Default target dataset: [TACO
(Trash Annotations in Context) — YOLO format](https://www.kaggle.com/datasets/vencerlanz09/taco-dataset-yolo-format),
real-scene litter photos with bounding boxes across dozens of fine-grained
categories.

**This dataset's exact category scheme needs verifying after download** —
same discipline as step 3. Run the inspection cell below, look at the
printed class names/ids, then fill in `WASTE_CLASS_MAP` before splitting.
Don't skip straight to training on an unverified guess.

In [ ]:
WASTE_RAW_DIR = f'{PROJECT_ROOT}/data/waste_raw'
import os
os.makedirs(WASTE_RAW_DIR, exist_ok=True)

!kaggle datasets download -d vencerlanz09/taco-dataset-yolo-format -p {WASTE_RAW_DIR} --unzip
!find {WASTE_RAW_DIR} -maxdepth 3 | head -30

In [ ]:
from src.preprocessing.dataset_prep import discover_pairs, inspect_class_distribution, find_class_names

waste_pairs = discover_pairs(Path(WASTE_RAW_DIR))
print(f'Found {len(waste_pairs)} image/label pairs')

waste_class_dist = inspect_class_distribution(waste_pairs)
waste_class_names_raw = find_class_names(Path(WASTE_RAW_DIR))
print('Class distribution:', waste_class_dist)
print('Discovered class names (id -> name, if a classes.txt/yaml was found):')
if waste_class_names_raw:
    for idx, name in enumerate(waste_class_names_raw):
        print(f'  {idx}: {name}  (count={waste_class_dist.get(idx, 0)})')
else:
    print('  No classes.txt/yaml found — cross-reference ids against the Kaggle dataset card manually.')

**Edit this before continuing.** `WASTE_CLASS_MAP` maps *source* class id ->
*output* class id, using the ids/names printed above. Any source id you
leave out of the map is dropped entirely (useful for categories with too
few examples to train on). The placeholder below groups into 5 practical
supercategories — adjust the keys to match what you actually saw printed,
and adjust `WASTE_CLASS_NAMES` to match the values you chose.

In [ ]:
# EDIT THESE TWO based on the class list printed above — this placeholder
# assumes a 0..N raw id scheme and will very likely need remapping.
WASTE_CLASS_NAMES = ['plastic', 'paper', 'metal_glass', 'organic', 'other_rubbish']
WASTE_CLASS_MAP = {
    # source_id: output_id
    0: 0, 1: 0,      # example: plastic bag, plastic bottle -> plastic
    2: 1, 3: 1,       # example: paper, cardboard -> paper
    4: 2, 5: 2,       # example: can, glass jar -> metal_glass
    6: 3,             # example: food waste -> organic
    7: 4,             # example: unlabeled litter -> other_rubbish
}
print('Using WASTE_CLASS_MAP:', WASTE_CLASS_MAP)

In [ ]:
from src.preprocessing.dataset_prep import split_dataset, write_data_yaml

WASTE_DIR = Path(PROJECT_ROOT) / 'data' / 'waste'
split_dataset(waste_pairs, WASTE_DIR, train=0.7, val=0.2, test=0.1, seed=42, class_id_map=WASTE_CLASS_MAP)
waste_yaml = write_data_yaml(WASTE_DIR / 'data.yaml', WASTE_DIR, names=WASTE_CLASS_NAMES)
print('Waste data.yaml:', waste_yaml)

Train the waste model — same wrapper as the vegetation model, just pointed
at a different `data.yaml`. A smaller/faster backbone (`yolov8n`) is a
reasonable default here since this is a secondary model, not the project's
headline ablation; bump to `yolov8s` if you have GPU time to spare.

In [ ]:
waste_cfg = TrainConfig(
    data_yaml=str(waste_yaml),
    model='yolov8n.pt',
    epochs=80,
    imgsz=512,
    batch=16,
    project=RUNS_DIR,
    name='waste',
)
waste_model, waste_results = train(waste_cfg)
waste_weights = f'{RUNS_DIR}/waste/weights/best.pt'

## 9. Combined inference — vegetation + waste together

Runs both models independently on the same original image and merges the
two detection sets into one annotated view: green boxes for vegetation,
red boxes for waste categories. This is the "complete scene" artifact —
one image, both models, nothing retrained or merged at the weights level.

In [ ]:
from src.inference.combined_predict import predict_combined_and_save

combined_demo_images = list((BASELINE_DIR / 'images' / 'test').iterdir())[:5]

for img_path in combined_demo_images:
    out_path = f'{PROJECT_ROOT}/reports/combined/{img_path.stem}_combined.jpg'
    predict_combined_and_save(
        img_path, masked_weights, waste_weights, out_path,
        veg_conf=0.25, waste_conf=0.25,
        veg_class_names=['green_vegetation'],
        waste_class_names=WASTE_CLASS_NAMES,
        veg_green_ratio_threshold=0.15,
        mask_config=mask_config,
    )
print('Saved combined annotations to', f'{PROJECT_ROOT}/reports/combined/')

In [ ]:
import matplotlib.pyplot as plt
import cv2

combined_preds = sorted(Path(f'{PROJECT_ROOT}/reports/combined').glob('*_combined.jpg'))
fig, axes = plt.subplots(1, len(combined_preds), figsize=(4 * len(combined_preds), 4))
for ax, p in zip(axes, combined_preds):
    ax.imshow(cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB))
    ax.set_title(p.stem, fontsize=8)
    ax.axis('off')
fig.tight_layout()
plt.show()

## 10. Everything is already on Drive

Since `PROJECT_ROOT` lives under `/content/drive/MyDrive/...`, weights (`runs/*/weights/best.pt`), plots (`reports/`), and the assembled datasets all persist automatically across Colab sessions — no extra export step needed.

In [ ]:
print('Baseline weights:', f'{baseline_run_dir}/weights/best.pt')
print('Masked weights:  ', f'{masked_run_dir}/weights/best.pt')
print('Reports:         ', f'{PROJECT_ROOT}/reports/')
print()
print('Final metrics summary:')
summary
print('Waste weights:     ', f'{RUNS_DIR}/waste/weights/best.pt')

---

# Part 2 — Leaf Detection System for Robotic Collection (Track B)

Everything above (Part 1) is a complete, finished result: HSV green-masking
vs baseline ablation for vegetation detection. **Nothing below touches it.**

This part builds a *separate* system per the robotic-collection spec:
single class `"leaf"`, trained on a **synthetically composited** dataset
(real leaf cutouts + real waste clutter + real harvested backgrounds — no
usable public dataset exists for "fallen leaves mixed with campus waste",
confirmed by search before building this). Green-masking doesn't apply
here — dry leaves aren't green — so this is plain-RGB YOLO training from
scratch, reusing the same `train.py` / `predict.py` / `metrics.py`
machinery from Part 1 wherever it's generic enough to fit.

**Local-disk staging**: cutout extraction, background harvesting, and the
generated synthetic dataset all write to `/content/track_b_work` (the
Colab VM's local disk) instead of Drive. Every read/write against a
Drive-mounted path goes through a network round-trip (Drive FUSE), and
training re-reads every image every epoch — pointing training at Drive
directly turns each epoch into hundreds of network calls instead of local
disk reads. `RUNS_DIR` (checkpoint output) stays on Drive as before —
those writes happen once per epoch, not once per image, so the cost is
negligible and persistence matters there. The synthetic dataset itself is
deterministically regenerable (same seed, same source cutouts/backgrounds)
so losing `/content` on a runtime reset just means re-running B1-B4, not
losing anything irreplaceable — an optional one-shot zip-to-Drive backup
is included at the end of B4 if you'd rather not regenerate.

**Scope for this pass**: Stages 1-4 from the spec (dataset generation → YOLO
leaf detection → image/video inference → counting/tracking) — the part
that's actually gradable. Camera calibration, coordinate transforms, target
selection, the robot command API, and the dashboard are deliberately
deferred until the detector itself is validated; building those against a
model that doesn't exist yet would be scaffolding for its own sake.

## B1. Leaf cutout source — Roboflow leaf-segmentation dataset

We need leaf images with **pixel masks**, not just boxes, so cutouts have
clean silhouettes instead of rectangular seams when pasted. Two small,
free Roboflow Universe projects fit:
[Leaf segmentation (Giovi)](https://universe.roboflow.com/giovi/leaf-segmentation-uxlob),
[Leaf Segmentation (PHD UTM)](https://universe.roboflow.com/phd-utm/leaf-segmentation-rtwov).

**Store your API key as a Colab Secret, never in cell text.** Click the key
icon (🔑) in the left sidebar → *Add new secret* → name it
`ROBOFLOW_API_KEY` → paste your key there → toggle notebook access on.
Secrets never get written into the saved notebook, so they survive
Colab's GitHub auto-save without ever reaching your repo — unlike pasting
a key directly into a cell, which *does* get committed the moment Colab
syncs back to GitHub (this happened once already in this project; the key
was rotated).

Get your project/version numbers from Roboflow's *Download this Dataset* →
format **YOLOv8** → *show download code* on each project page (this export
format gives polygon labels, which is what `cutout_extractor.py`
expects) — copy just the workspace/project/version values, not the whole
snippet with its embedded key.

In [ ]:
!pip install -q roboflow

from google.colab import userdata
from roboflow import Roboflow

LOCAL_WORK_DIR = '/content/track_b_work'
LEAF_SEG_RAW_DIR = f'{LOCAL_WORK_DIR}/leaf_seg_raw'
import os
os.makedirs(LEAF_SEG_RAW_DIR, exist_ok=True)

rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))

# Fill in workspace/project/version from each project's download page —
# just the identifiers, the key itself comes from the secret above.
LEAF_SEG_PROJECTS = [
    # (workspace, project_slug, version_number, subfolder_name)
    ('giovi', 'leaf-segmentation-uxlob', 4, 'giovi'),  # verified via the project page's displayed model id
    # ('phd-utm', 'leaf-segmentation-rtwov', 1, 'phd_utm'),
]

for workspace, project_slug, version_num, subfolder in LEAF_SEG_PROJECTS:
    project = rf.workspace(workspace).project(project_slug)
    project.version(version_num).download('yolov8', location=f'{LEAF_SEG_RAW_DIR}/{subfolder}')

**If a downloaded project turns out to have plain boxes instead of
polygons** (some Roboflow "segmentation" exports fall back to boxes if the
source annotations weren't actually polygon-drawn), the cutout builder
below will just extract fewer/no instances from it — check the printed
count and swap in `auto_segment_plain_background` from
`cutout_extractor.py` on that subset if needed, rather than assuming
something is broken.

In [ ]:
from pathlib import Path
from src.synthetic.cutout_extractor import extract_cutouts_from_yolo_seg_dataset

LEAF_CUTOUT_DIR = Path(LOCAL_WORK_DIR) / 'leaf_cutouts'
leaf_cutout_paths = extract_cutouts_from_yolo_seg_dataset(Path(LEAF_SEG_RAW_DIR), LEAF_CUTOUT_DIR)
print(f'Extracted {len(leaf_cutout_paths)} leaf cutouts to {LEAF_CUTOUT_DIR}')

## B2. Waste cutout source — reuse the TACO download from Part 1

If you already ran the waste-detection section in Part 1, `WASTE_RAW_DIR`
is already populated — no new download needed. These cutouts are pasted
purely as **unlabeled visual clutter** (the spec is explicit: only `leaf`
is a detection class), so we don't need the class remapping from Part 1
here, just raw bounding-box crops.

In [ ]:
from src.preprocessing.dataset_prep import discover_pairs
from src.synthetic.cutout_extractor import cutout_from_bbox
import cv2

if 'WASTE_RAW_DIR' not in dir():
    WASTE_RAW_DIR = f'{PROJECT_ROOT}/data/waste_raw'  # from Part 1 section 8 — re-download there first if empty

waste_pairs_for_cutouts = discover_pairs(Path(WASTE_RAW_DIR))
WASTE_CUTOUT_DIR = Path(LOCAL_WORK_DIR) / 'waste_cutouts'
WASTE_CUTOUT_DIR.mkdir(parents=True, exist_ok=True)

waste_cutout_paths = []
for img_path, lbl_path in waste_pairs_for_cutouts[:400]:  # cap — we only need clutter variety, not the full dataset
    image = cv2.imread(str(img_path))
    if image is None:
        continue
    h, w = image.shape[:2]
    for i, line in enumerate(lbl_path.read_text().splitlines()):
        if not line.strip():
            continue
        _, cx, cy, bw, bh = line.split()[:5]
        cx, cy, bw, bh = float(cx)*w, float(cy)*h, float(bw)*w, float(bh)*h
        box = (cx-bw/2, cy-bh/2, cx+bw/2, cy+bh/2)
        try:
            cutout = cutout_from_bbox(image, box)
        except ValueError:
            continue
        out_path = WASTE_CUTOUT_DIR / f'{img_path.stem}_{i}.png'
        cv2.imwrite(str(out_path), cutout)
        waste_cutout_paths.append(out_path)

print(f'Extracted {len(waste_cutout_paths)} waste cutouts to {WASTE_CUTOUT_DIR}')

## B3. Background source — harvest from datasets we already have

No public "empty campus ground" dataset exists (checked before building
this). Instead we harvest object-free patches directly from data we
already downloaded: soil-only regions of the Part-1 crop/weed dataset
(inverted green mask, away from any labeled box) and non-litter regions of
the TACO images. Both are real photos of real outdoor ground.

In [ ]:
from src.synthetic.background_harvester import harvest_from_green_dataset, harvest_from_boxed_dataset
from src.preprocessing.hsv_mask import MaskConfig

BACKGROUND_DIR = Path(LOCAL_WORK_DIR) / 'backgrounds'

green_ds_pairs = discover_pairs(Path(RAW_DATA_DIR))  # Part 1's crop/weed raw data
bg_from_green = harvest_from_green_dataset(
    green_ds_pairs, BACKGROUND_DIR / 'from_crop_weed', patch_size=256, patches_per_image=2,
    mask_config=mask_config if 'mask_config' in dir() else MaskConfig(), max_source_images=150,
)

bg_from_waste = harvest_from_boxed_dataset(
    waste_pairs_for_cutouts, BACKGROUND_DIR / 'from_taco', patch_size=256, patches_per_image=1, max_source_images=150,
)

background_paths = bg_from_green + bg_from_waste
print(f'Harvested {len(background_paths)} background patches ({len(bg_from_green)} soil, {len(bg_from_waste)} scene)')

## B4. Generate the synthetic training set

This is the "Controlled Synthetic Generation" step — paste cutouts onto
backgrounds with randomized geometry across easy/medium/hard difficulty
tiers. Bounding boxes come out of the paste operation automatically, so
there's no manual annotation step at all. **Look at the sanity-check grid
below before committing GPU time to training** — if leaves look obviously
fake (wrong scale, floating without shadow, always centered), tune
`n_leaves_range` / `DIFFICULTY_PRESETS` in `src/synthetic/compositor.py`
first.

In [ ]:
from src.synthetic.generate_dataset import generate_synthetic_dataset

LEAF_DATASET_DIR = Path(LOCAL_WORK_DIR) / 'leaf_synthetic'  # local — read every training epoch, must not be on Drive
generate_synthetic_dataset(
    leaf_cutout_paths, background_paths, LEAF_DATASET_DIR,
    waste_cutout_paths=waste_cutout_paths,
    n_train=900, n_val=150, n_test=150,
    n_leaves_range=(2, 8), canvas_size=(640, 640), seed=42,
)
leaf_data_yaml = LEAF_DATASET_DIR / 'data.yaml'
print('Leaf data.yaml:', leaf_data_yaml)

In [ ]:
import cv2
import matplotlib.pyplot as plt

sample_imgs = sorted((LEAF_DATASET_DIR / 'images' / 'train').iterdir())[:6]
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, img_path in zip(axes.flat, sample_imgs):
    image = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    h, w = image.shape[:2]
    lbl_path = LEAF_DATASET_DIR / 'labels' / 'train' / img_path.with_suffix('.txt').name
    for line in lbl_path.read_text().splitlines():
        if not line.strip():
            continue
        _, cx, cy, bw, bh = [float(v) for v in line.split()]
        x1, y1 = int((cx - bw/2) * w), int((cy - bh/2) * h)
        x2, y2 = int((cx + bw/2) * w), int((cy + bh/2) * h)
        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
    ax.imshow(image)
    ax.set_title(img_path.stem, fontsize=8)
    ax.axis('off')
fig.tight_layout()
plt.show()

**Optional** — back up the generated dataset to Drive as a single zip
(one bulk write, not 1200+ small ones, so it's fast). Skip this if you're
fine regenerating from B1-B4 next session — it's deterministic (same seed,
same source cutouts) as long as the cutout/background pools haven't
changed.

In [ ]:
import shutil

BACKUP_TO_DRIVE = False  # flip to True if you want a persisted copy

if BACKUP_TO_DRIVE:
    zip_base = f'{PROJECT_ROOT}/data/leaf_synthetic_backup'
    shutil.make_archive(zip_base, 'zip', LEAF_DATASET_DIR)
    print('Backed up to', zip_base + '.zip')
else:
    print('Skipped — regenerate via B1-B4 next session if needed.')

## B5. Train the leaf detector

Same wrapper as Part 1, pointed at the synthetic dataset, single class
`leaf`. `yolov8s` is a reasonable default; drop to `yolov8n` if Colab's
free-tier GPU queue is tight on time.

In [ ]:
leaf_cfg = TrainConfig(
    data_yaml=str(leaf_data_yaml),
    model='yolov8s.pt',
    epochs=100,
    imgsz=640,
    batch=16,
    project=RUNS_DIR,
    name='leaf',
)
leaf_model, leaf_results = train(leaf_cfg)
leaf_weights = f'{RUNS_DIR}/leaf/weights/best.pt'

## B6. Evaluate

In [ ]:
leaf_run_dir = f'{RUNS_DIR}/leaf'
leaf_df = load_results_csv(leaf_run_dir)
print(summarize_final_metrics(leaf_df))
plot_loss_curves(leaf_df, f'{PROJECT_ROOT}/reports/leaf_loss.png', 'Leaf Detector — Loss')
plot_map_curves(leaf_df, f'{PROJECT_ROOT}/reports/leaf_map.png', 'Leaf Detector — mAP')

## B7. Image inference — spec-shaped JSON output

This is the interface contract a future robot-side component would
consume (spec Section 20). No robot code exists yet, but the output shape
already matches what one would need.

In [ ]:
import json
from src.inference.track import predict_image_response

test_leaf_images = list((LEAF_DATASET_DIR / 'images' / 'test').iterdir())[:3]
for img_path in test_leaf_images:
    response = predict_image_response(leaf_weights, img_path, conf=0.25, class_names=['leaf'])
    print(img_path.name, '->', json.dumps(response, indent=2))

## B8. Video tracking demo

Upload a short clip if you have one (leaves/ground, a few seconds is
enough). If not, the cell below builds a small synthetic demo clip by
panning the compositor across a few frames, purely so the tracking API can
be exercised end-to-end without needing real footage yet.

In [ ]:
from google.colab import files

UPLOAD_REAL_VIDEO = False  # flip to True if you're uploading your own clip

if UPLOAD_REAL_VIDEO:
    uploaded = files.upload()
    video_path = Path(PROJECT_ROOT) / 'data' / list(uploaded.keys())[0]
    video_path.write_bytes(list(uploaded.values())[0])
else:
    from src.synthetic.compositor import compose_scene
    video_path = Path(PROJECT_ROOT) / 'data' / 'synthetic_demo_clip.mp4'
    bg = cv2.imread(str(background_paths[0]))
    writer = cv2.VideoWriter(str(video_path), cv2.VideoWriter_fourcc(*'mp4v'), 5, (bg.shape[1], bg.shape[0]))
    for i in range(20):
        demo_cutouts = [cv2.imread(str(p), cv2.IMREAD_UNCHANGED) for p in leaf_cutout_paths[:5]]
        result = compose_scene(bg, demo_cutouts, n_leaves=(3, 3), difficulty='easy', seed=i)
        writer.write(result.image)
    writer.release()
    print('Synthetic demo clip (not real footage) written to', video_path)

In [ ]:
from src.inference.track import track_video_to_json

track_out = track_video_to_json(leaf_weights, str(video_path), f'{PROJECT_ROOT}/reports/leaf_tracks.json', conf=0.25, class_names=['leaf'])
print('Per-frame tracking results saved to', track_out)

import json
frames = json.loads(track_out.read_text())
print(f'{len(frames)} frames tracked. Sample frame:')
print(json.dumps(frames[len(frames)//2], indent=2))

## B9. Everything is on Drive

Same pattern as Part 1 — everything under `PROJECT_ROOT` persists to
Google Drive automatically:

- `data/leaf_cutouts/`, `data/waste_cutouts/`, `data/backgrounds/` — the
  reusable source material for regenerating or extending the synthetic set
- `data/leaf_synthetic/` — the generated training set itself
- `runs/leaf/weights/best.pt` — the trained leaf detector
- `reports/leaf_tracks.json` — the tracking demo output

**Deferred to a later pass** (per the "CV core first" scope decision):
camera calibration, coordinate transforms, target selection, the mock
robot command API, the state machine, and the dashboard. Nothing here
blocks adding them once the detector's real-world accuracy is validated —
that was the whole point of building this before the robot-facing layers.

---

# Part 3 — Roadmap Improvements

Everything below is additive and non-destructive: it does not change
`baseline`, `masked`, `waste`, or the original `leaf` run — those are
already trained and already in your results review. New experiments get
new names (`leaf_v2`) so old and new stay comparable side by side rather
than one silently overwriting the other.

## C1. Unified Track B pipeline (optional — for future reruns)

`src/synthetic/pipeline.py` wraps B1-B4 (cutout extraction, background
harvesting, synthetic generation) behind one call. You don't need to
re-run this now — your existing `leaf_cutout_paths`, `background_paths`,
and `LEAF_DATASET_DIR` from B1-B4 are already built. This is for next
time you rebuild the dataset from scratch (e.g. after adding a second
leaf-segmentation source), so it's one call instead of wiring six cells
together by hand.

In [ ]:
from src.synthetic.pipeline import build_leaf_dataset

# Example only — not run now, since B1-B4 already built everything it produces:
#
# result = build_leaf_dataset(
#     leaf_seg_raw_dir=LEAF_SEG_RAW_DIR,
#     veg_dataset_pairs=green_ds_pairs,
#     waste_dataset_pairs=waste_pairs_for_cutouts,
#     work_dir=LOCAL_WORK_DIR,
#     n_train=900, n_val=150, n_test=150,
# )
# leaf_data_yaml = result.data_yaml
# LEAF_DATASET_DIR = result.dataset_dir  # <- alias so C4/C6/the backup cell below all work unchanged
print('build_leaf_dataset available — see commented example above.')

## C2. Augmented training run — `leaf_v2`

Adds mixup and random perspective to the existing HSV/mosaic/rotation
augmentation already used for `leaf` — the two missing pieces from the
roadmap's augmentation ask. Same dataset as the original `leaf` run
(`leaf_data_yaml`), so augmentation is isolated as the only variable,
same principle as the baseline-vs-masked ablation in Part 1. This trains
a full new model — budget real GPU time, same as the original leaf run.

In [ ]:
leaf_v2_cfg = TrainConfig(
    data_yaml=str(leaf_data_yaml),
    model='yolov8s.pt',
    epochs=100,
    imgsz=640,
    batch=16,
    project=RUNS_DIR,
    name='leaf_v2',
    mixup=0.15,
    perspective=0.0003,
)
leaf_v2_model, leaf_v2_results = train(leaf_v2_cfg)
leaf_v2_weights = f'{RUNS_DIR}/leaf_v2/weights/best.pt'

## C3. Compare `leaf` vs `leaf_v2`

In [ ]:
leaf_v2_run_dir = f'{RUNS_DIR}/leaf_v2'
leaf_v2_df = load_results_csv(leaf_v2_run_dir)

print('leaf    (original): ', summarize_final_metrics(leaf_df))
print('leaf_v2 (augmented):', summarize_final_metrics(leaf_v2_df))

compare_runs(leaf_run_dir, leaf_v2_run_dir, f'{PROJECT_ROOT}/reports/leaf_vs_leaf_v2.png')

## C4. Test-time augmentation — production inference

`augment=True` runs multi-scale + flip inference and averages the
result — catches small/marginal detections at ~2-3x slower inference.
Meant for final evaluation or a one-off careful pass, not live video.

In [ ]:
from src.inference.track import predict_image_response

sample_img = test_leaf_images[0] if 'test_leaf_images' in dir() else list((LEAF_DATASET_DIR / 'images' / 'test').iterdir())[0]

standard = predict_image_response(leaf_v2_weights, sample_img, conf=0.25, class_names=['leaf'], augment=False)
tta = predict_image_response(leaf_v2_weights, sample_img, conf=0.25, class_names=['leaf'], augment=True)

print(f'Standard inference: {standard["leaf_count"]} leaves')
print(f'TTA inference:      {tta["leaf_count"]} leaves')

## C5. Temporally-smoothed tracking

Same tracking demo as B8, with `smooth=True` — applies an exponential
moving average to each tracked leaf's center coordinates (see
`TrackSmoother` in `src/inference/track.py`) so the coordinates fed to a
future robot controller don't jitter frame-to-frame.

In [ ]:
from src.inference.track import track_video_to_json

smoothed_track_out = track_video_to_json(
    leaf_v2_weights, str(video_path), f'{PROJECT_ROOT}/reports/leaf_tracks_smoothed.json',
    conf=0.25, class_names=['leaf'], smooth=True, smooth_alpha=0.4,
)
print('Smoothed tracking results saved to', smoothed_track_out)

## C6. Low-confidence error logging

Runs the leaf detector over the test split and logs every image with a
detection under the threshold — an annotated snapshot plus a JSONL index
entry per case. This is the active-learning queue: images worth
relabeling or adding to a future training round.

In [ ]:
from src.inference.error_logger import LowConfidenceLogger
from src.inference.predict import predict_image

logger = LowConfidenceLogger(f'{PROJECT_ROOT}/reports/low_confidence_log', threshold=0.5, class_names=['leaf'])

test_images = list((LEAF_DATASET_DIR / 'images' / 'test').iterdir())
for img_path in test_images:
    image, detections = predict_image(leaf_v2_weights, img_path, conf=0.25)
    logger.log(image, detections, img_path.name)

print('Low-confidence log summary:', logger.summary())

## C7. Real-world field validation (ready, not run)

This needs real photos, which don't exist yet in this project — nothing
here can substitute for them. The moment you have some (even 15-20 real
campus photos of leaves/waste), upload them into
`{PROJECT_ROOT}/data/field_validation/images/` (and optionally a matching
`labels/` folder in YOLO format if you've hand-labeled any), then run the
cell below. Without labels you get detection-rate and confidence
statistics; with labels you get real, measured mAP/precision/recall —
the actual number that would close the "Track B has no real-world
validation" limitation from your results review.

In [ ]:
from src.evaluation.field_validation import run_field_validation

FIELD_DIR = Path(PROJECT_ROOT) / 'data' / 'field_validation'
(FIELD_DIR / 'images').mkdir(parents=True, exist_ok=True)

n_images = len(list((FIELD_DIR / 'images').glob('*')))
if n_images == 0:
    print(f'No images yet in {FIELD_DIR / "images"} — upload real photos there, then re-run this cell.')
else:
    field_result = run_field_validation(
        leaf_v2_weights, FIELD_DIR, f'{PROJECT_ROOT}/reports/field_validation',
        conf=0.25, class_names=['leaf'],
    )
    print(json.dumps(field_result, indent=2))

## C8. Everything from Part 3 is on Drive

`runs/leaf_v2/weights/best.pt`, the comparison figure, the smoothed
tracking JSON, the low-confidence log, and field validation output (once
real photos exist) all write under `PROJECT_ROOT` — same persistence
guarantee as everything else in this notebook.

## C9. Test your own photos — full pipeline, image in to final output

Upload any photo and see the complete path: image → model → annotated
result + the exact spec-shaped JSON a future robot-side component would
consume. No masking step here (that's Track A's thing, doesn't apply to
leaves) — the model runs directly on what you upload.

In [ ]:
from google.colab import files

leaf_upload_dir = Path(PROJECT_ROOT) / 'data' / 'leaf_custom_test'
leaf_upload_dir.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()  # pick photos from your device
custom_leaf_images = []
for fname, content in uploaded.items():
    dest = leaf_upload_dir / fname
    dest.write_bytes(content)
    custom_leaf_images.append(dest)

print(f'Uploaded {len(custom_leaf_images)} image(s)')

In [ ]:
from src.inference.predict import predict_image, draw_detections
from src.inference.track import predict_image_response
import json

fig, axes = plt.subplots(1, len(custom_leaf_images), figsize=(6 * len(custom_leaf_images), 6), squeeze=False)

for ax, img_path in zip(axes[0], custom_leaf_images):
    image, detections = predict_image(leaf_v2_weights, img_path, conf=0.25)
    annotated = draw_detections(image, detections, class_names=['leaf'])
    ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    ax.set_title(f'{img_path.name} — {len(detections)} leaves', fontsize=10)
    ax.axis('off')

    response = predict_image_response(leaf_v2_weights, img_path, conf=0.25, class_names=['leaf'])
    print(f'--- {img_path.name} ---')
    print(json.dumps(response, indent=2))

fig.tight_layout()
plt.show()

If detections look sparse or missing on a real photo (expected — the model
only ever saw synthetic composited scenes), try `conf=0.1` and
`augment=True` (TTA) before concluding the model failed — that's a
distribution-shift finding worth noting for the "no real-world
validation yet" limitation, not necessarily a bug.

## C10. Regenerate synthetic dataset with a fixed compositor — `leaf_v3`

`leaf` and `leaf_v2` were both trained on a dataset built with
`n_leaves_range=(2, 8)` passed explicitly to `generate_synthetic_dataset`
(see B5) — that override applies uniformly to every difficulty tier,
which is why scenes topped out at 8 scattered, uniformly fresh-green
leaves regardless of the `pile` tier's own density preset. That's the
root cause behind `leaf_v2` detecting nothing on real leaf-pile / litter
photos.

Since then `src/synthetic/compositor.py` gained: canvas-relative
`leaf_size_frac` (was accidentally tied to source cutout resolution),
Gaussian mound clustering for `pile` scenes, dry/brown HSV tinting, and a
new `scatter` tier — independently-placed (non-clustered) leaves at
moderate density, matching real reference photos of campus paths where
leaves land individually rather than in mounds. `DEFAULT_DIFFICULTY_MIX`
now weights `scatter` at 35%, the largest single share, since that's the
dominant pattern in the real photos this project is actually being
tested against.

This cell reuses the cutouts and backgrounds already built in B1-B4 — no
re-download, no GPU needed, just re-running the paste/composite step.
Critically, `n_leaves_range` is left `None` here (unlike B5) so every
tier's own density preset actually takes effect.

In [ ]:
from src.synthetic.generate_dataset import generate_synthetic_dataset

LEAF_DATASET_V3_DIR = Path(LOCAL_WORK_DIR) / 'leaf_synthetic_v3'  # local — must not be on Drive
generate_synthetic_dataset(
    leaf_cutout_paths, background_paths, LEAF_DATASET_V3_DIR,
    waste_cutout_paths=waste_cutout_paths,
    n_train=900, n_val=150, n_test=150,
    canvas_size=(640, 640), seed=42,
    # n_leaves_range intentionally omitted — each difficulty tier
    # (including the new 'pile' and 'scatter') uses its own density.
)
leaf_v3_data_yaml = LEAF_DATASET_V3_DIR / 'data.yaml'
print('leaf_v3 data.yaml:', leaf_v3_data_yaml)

## C10b. Visual sanity check — look before you train

Same principle as the local placeholder check done before this dataset
was regenerated: confirm `pile` actually reads as a mound and `scatter`
actually reads as independently-placed leaves, before spending a GPU
budget on training. Pick a few samples per difficulty tier from the file
name (`{split}_{i:05d}_{difficulty}.jpg`).

In [ ]:
import random

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
train_imgs = sorted((LEAF_DATASET_V3_DIR / 'images' / 'train').iterdir())
by_difficulty = {}
for p in train_imgs:
    diff = p.stem.rsplit('_', 1)[-1]
    by_difficulty.setdefault(diff, []).append(p)

for ax, (diff, paths) in zip(axes.flat, sorted(by_difficulty.items())):
    img_path = random.choice(paths)
    image = cv2.imread(str(img_path))
    lbl_path = LEAF_DATASET_V3_DIR / 'labels' / 'train' / img_path.with_suffix('.txt').name
    h, w = image.shape[:2]
    for line in lbl_path.read_text().splitlines():
        _, cx, cy, bw, bh = map(float, line.split())
        x0, y0 = int((cx - bw / 2) * w), int((cy - bh / 2) * h)
        x1, y1 = int((cx + bw / 2) * w), int((cy + bh / 2) * h)
        cv2.rectangle(image, (x0, y0), (x1, y1), (0, 0, 255), 2)
    ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    ax.set_title(f'{diff} — {img_path.name}', fontsize=10)
    ax.axis('off')

fig.tight_layout()
plt.show()

## C11. Train `leaf_v3`

Same augmentation config as `leaf_v2` (mixup + perspective) — the only
variable changed here is the dataset itself, so any improvement is
attributable to the compositor fixes, not a confound with training
hyperparameters.

In [ ]:
leaf_v3_cfg = TrainConfig(
    data_yaml=str(leaf_v3_data_yaml),
    model='yolov8s.pt',
    epochs=100,
    imgsz=640,
    batch=16,
    project=RUNS_DIR,
    name='leaf_v3',
    mixup=0.15,
    perspective=0.0003,
)
leaf_v3_model, leaf_v3_results = train(leaf_v3_cfg)
leaf_v3_weights = f'{RUNS_DIR}/leaf_v3/weights/best.pt'

## C12. Field validation — `leaf_v3` vs `leaf_v2` on real photos

This is the actual test that matters: does the compositor fix translate
into real-world detections? `FIELD_DIR` (`data/field_validation/images/`)
already has one confirmed-real seed photo (`green_leaf_01.jpg`) copied in
locally — **add more real camera photos there before running this cell**,
the more the better (10-30+). AI-generated reference images and stock
photos should NOT go in this folder: they'd inflate/distort the one
number meant to measure genuine real-world performance.

Runs both `leaf_v2` and `leaf_v3` over the same real photos so the
before/after is directly comparable, not just `leaf_v3` in isolation.

In [ ]:
from src.evaluation.field_validation import run_field_validation

FIELD_DIR = Path(PROJECT_ROOT) / 'data' / 'field_validation'
(FIELD_DIR / 'images').mkdir(parents=True, exist_ok=True)

n_images = len(list((FIELD_DIR / 'images').glob('*')))
if n_images == 0:
    print(f'No images yet in {FIELD_DIR / "images"} — upload real photos there, then re-run this cell.')
else:
    print(f'Validating on {n_images} real photo(s)...')
    v2_result = run_field_validation(
        leaf_v2_weights, FIELD_DIR, f'{PROJECT_ROOT}/reports/field_validation_v2',
        conf=0.25, class_names=['leaf'],
    )
    v3_result = run_field_validation(
        leaf_v3_weights, FIELD_DIR, f'{PROJECT_ROOT}/reports/field_validation_v3',
        conf=0.25, class_names=['leaf'],
    )
    print('leaf_v2:', json.dumps(v2_result, indent=2))
    print('leaf_v3:', json.dumps(v3_result, indent=2))

## C12b. Pipeline figure for the report — `leaf_v3`

Track A's report figure shows Original → Green Mask → Masked →
Detection → Leaf Count. Track B has no masking stage (it trains and
infers on plain RGB), so the equivalent figure here is Original Input →
Model Input (letterboxed to imgsz — the actual resize-and-pad transform
YOLO applies before the image reaches the network) → Final Detection.

Runs inference only against the already-trained `leaf_v3_weights` —
**no training happens in this cell.**

Picks up every real photo currently in `data/field_validation/images/`
plus a couple of synthetic test-split images for density contrast, and
labels each row's actual provenance explicitly (`REAL PHOTO` vs
`SYNTHETIC`) — do not remove that labeling; it's there specifically so
this figure can't misrepresent a synthetic or AI-generated image as a
real photo in the report/presentation.

In [ ]:
from src.evaluation.report import plot_leaf_pipeline_grid

FIELD_DIR = Path(PROJECT_ROOT) / 'data' / 'field_validation'
real_photos = sorted(
    p for p in (FIELD_DIR / 'images').iterdir()
    if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}
) if (FIELD_DIR / 'images').exists() else []

# LEAF_DATASET_V3_DIR/LEAF_DATASET_DIR only exist if you've run the dataset
# -build cells (B1-B4/C10) this session — fall back to the known fixed local
# -disk paths rather than requiring a re-run just for this figure.
_local_work_dir = globals().get('LOCAL_WORK_DIR', '/content/track_b_work')
_dataset_dir_candidates = [
    globals().get('LEAF_DATASET_V3_DIR'),
    globals().get('LEAF_DATASET_DIR'),
    Path(_local_work_dir) / 'leaf_synthetic_v3',
    Path(_local_work_dir) / 'leaf_synthetic',
]
synth_samples = []
for cand in _dataset_dir_candidates:
    if cand is None:
        continue
    test_dir = Path(cand) / 'images' / 'test'
    if test_dir.exists():
        synth_samples = sorted(test_dir.iterdir())[:2]
        break

if not synth_samples:
    print('No synthetic test-split images found this session (dataset dir not built/loaded) — using real photos only.')

sample_paths = real_photos + synth_samples
row_labels = [f'REAL PHOTO — {p.name}' for p in real_photos] + \
             [f'SYNTHETIC — {p.name}' for p in synth_samples]

if not sample_paths:
    raise RuntimeError('No images available — add real photos to data/field_validation/images/, '
                        'or run the dataset-generation cell (C10) first to rebuild the synthetic test split.')

pipeline_fig_path = plot_leaf_pipeline_grid(
    sample_paths, leaf_v3_weights, f'{PROJECT_ROOT}/reports/leaf_v3_pipeline_grid.png',
    row_labels=row_labels, conf=0.25, class_names=['leaf'],
)
print(f'{len(real_photos)} real photo(s), {len(synth_samples)} synthetic sample(s) — saved to {pipeline_fig_path}')

from PIL import Image as PILImage
display(PILImage.open(pipeline_fig_path))

## C13. Model-assisted pre-labeling for real photos (optional, speeds up annotation)

Drawing every box from scratch is slow. This runs `leaf_v3` over
`FIELD_DIR/images` and writes its own predictions as YOLO `.txt` files
into `FIELD_DIR/labels/` — a starting point to correct in any annotation
tool (LabelImg, CVAT, Roboflow, makesense.ai), not ground truth. It only
writes labels for images that don't already have one, so re-running this
never overwrites labels you've already hand-corrected.

**Do not skip straight to C14 after this** — review and fix every box
first. Training on uncorrected model proposals just teaches the model to
repeat its own mistakes more confidently.

In [ ]:
from src.inference.predict import predict_image

FIELD_DIR = Path(PROJECT_ROOT) / 'data' / 'field_validation'
(FIELD_DIR / 'images').mkdir(parents=True, exist_ok=True)
(FIELD_DIR / 'labels').mkdir(parents=True, exist_ok=True)

field_images = sorted(p for p in (FIELD_DIR / 'images').iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png'})
pre_labeled = 0
for img_path in field_images:
    label_path = FIELD_DIR / 'labels' / img_path.with_suffix('.txt').name
    if label_path.exists():
        continue  # never overwrite existing (possibly hand-corrected) labels
    image, detections = predict_image(leaf_v3_weights, img_path, conf=0.25)
    h, w = image.shape[:2]
    lines = []
    for det in detections:
        x0, y0, x1, y1 = det.box
        cx, cy = (x0 + x1) / 2 / w, (y0 + y1) / 2 / h
        bw, bh = (x1 - x0) / w, (y1 - y0) / h
        lines.append(f'0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}')
    label_path.write_text('\n'.join(lines) + ('\n' if lines else ''))
    pre_labeled += 1

print(f'Pre-labeled {pre_labeled} image(s) (proposals only) in {FIELD_DIR / "labels"}.')
print(f'{len(field_images) - pre_labeled} image(s) already had labels and were left untouched.')
print('Review and correct every box before running C14.')

## C14. Fine-tune `leaf_v3` on real photos — `leaf_v3_ft`

This is the highest-leverage step in this whole notebook for closing the
real-world gap: a small amount of real labeled data, fine-tuned on top of
the synthetic-trained model, closes sim-to-real gaps far more reliably
than any further synthetic-compositor tuning. Only run this after the
labels in `FIELD_DIR/labels/` are correct (from C13 + manual review, or
hand-labeled from scratch).

With fewer than 10 labeled real photos there's no meaningful held-out
split, so this fine-tunes on everything available and C15 falls back to
a visual check instead of a numeric metric — that's expected until more
real photos exist, not a bug. With 10+, it holds out 20% for a genuine
measured comparison in C15.

Augmentation is turned off here (unlike `leaf_v2`/`leaf_v3`) — the real
set is tiny, so the goal is gentle adaptation to real-photo statistics,
not more synthetic-style augmentation on top of already-real images.

In [ ]:
import random
import shutil

import yaml

FIELD_DIR = Path(PROJECT_ROOT) / 'data' / 'field_validation'  # self-contained: works even if C12/C13 weren't run this session
FIELD_LABELS_DIR = FIELD_DIR / 'labels'
labeled_images = sorted(
    p for p in (FIELD_DIR / 'images').iterdir()
    if p.suffix.lower() in {'.jpg', '.jpeg', '.png'} and (FIELD_LABELS_DIR / p.with_suffix('.txt').name).exists()
)

if len(labeled_images) < 3:
    print(f'Only {len(labeled_images)} labeled real photo(s) in {FIELD_LABELS_DIR} — need at '
          f'least a handful to fine-tune. Run C13, correct the proposed labels (or hand-label '
          f'some real photos), then re-run this cell.')
else:
    rng = random.Random(42)
    shuffled = labeled_images[:]
    rng.shuffle(shuffled)
    HAS_HELDOUT_SPLIT = len(shuffled) >= 10
    if HAS_HELDOUT_SPLIT:
        n_val = max(2, int(len(shuffled) * 0.2))
        val_imgs, train_imgs = shuffled[:n_val], shuffled[n_val:]
    else:
        val_imgs, train_imgs = [], shuffled
        print(f'Only {len(shuffled)} labeled real photos — fine-tuning on all of them, no '
              f'held-out split (too few for a meaningful number). C15 will show annotated '
              f'overlays instead of a metric.')

    FT_DIR = Path(LOCAL_WORK_DIR) / 'leaf_finetune'
    for split, imgs in [('train', train_imgs), ('val', val_imgs or train_imgs)]:
        (FT_DIR / 'images' / split).mkdir(parents=True, exist_ok=True)
        (FT_DIR / 'labels' / split).mkdir(parents=True, exist_ok=True)
        for img_path in imgs:
            shutil.copy(img_path, FT_DIR / 'images' / split / img_path.name)
            shutil.copy(FIELD_LABELS_DIR / img_path.with_suffix('.txt').name,
                        FT_DIR / 'labels' / split / img_path.with_suffix('.txt').name)

    ft_data_yaml = FT_DIR / 'data.yaml'
    yaml.safe_dump({
        'path': str(FT_DIR.resolve()),
        'train': 'images/train',
        'val': 'images/val',
        'nc': 1,
        'names': ['leaf'],
    }, open(ft_data_yaml, 'w'), sort_keys=False)

    leaf_v3_ft_cfg = TrainConfig(
        data_yaml=str(ft_data_yaml),
        model=leaf_v3_weights,  # start from leaf_v3, not from scratch — this is a fine-tune
        epochs=40,
        imgsz=640,
        batch=min(8, len(train_imgs)),
        patience=15,
        project=RUNS_DIR,
        name='leaf_v3_ft',
        mixup=0.0,
        perspective=0.0,
    )
    leaf_v3_ft_model, leaf_v3_ft_results = train(leaf_v3_ft_cfg)
    leaf_v3_ft_weights = f'{RUNS_DIR}/leaf_v3_ft/weights/best.pt'

## C15. Evaluate `leaf_v3` vs `leaf_v3_ft` on real photos

With a held-out split (10+ labeled real photos, see C14): runs both
models on the same held-out real images never used for fine-tuning, for
a genuine, leak-free before/after comparison. With fewer real photos:
shows annotated overlays from `leaf_v3_ft` instead, since a numeric
metric on 1-2 held-out images would be noise, not signal.

In [ ]:
import json
import shutil

from src.evaluation.field_validation import run_field_validation
from src.inference.predict import predict_image, draw_detections

if 'leaf_v3_ft_weights' not in dir():
    print('C14 has not produced leaf_v3_ft_weights yet — run C13 (pre-label), correct the '
          'labels, then C14 (fine-tune) before this cell.')
elif HAS_HELDOUT_SPLIT:
    held_out_dir = Path(LOCAL_WORK_DIR) / 'leaf_finetune_val_eval'
    (held_out_dir / 'images').mkdir(parents=True, exist_ok=True)
    (held_out_dir / 'labels').mkdir(parents=True, exist_ok=True)
    for img_path in val_imgs:
        shutil.copy(img_path, held_out_dir / 'images' / img_path.name)
        shutil.copy(FIELD_LABELS_DIR / img_path.with_suffix('.txt').name,
                    held_out_dir / 'labels' / img_path.with_suffix('.txt').name)

    before = run_field_validation(leaf_v3_weights, held_out_dir,
                                   f'{PROJECT_ROOT}/reports/field_validation_v3_before_ft',
                                   conf=0.25, class_names=['leaf'])
    after = run_field_validation(leaf_v3_ft_weights, held_out_dir,
                                  f'{PROJECT_ROOT}/reports/field_validation_v3_after_ft',
                                  conf=0.25, class_names=['leaf'])
    print('leaf_v3 (before fine-tune) on held-out real photos:', json.dumps(before, indent=2))
    print('leaf_v3_ft (after fine-tune) on held-out real photos:', json.dumps(after, indent=2))
else:
    fig, axes = plt.subplots(1, len(train_imgs), figsize=(6 * len(train_imgs), 6), squeeze=False)
    for ax, img_path in zip(axes[0], train_imgs):
        image, detections = predict_image(leaf_v3_ft_weights, img_path, conf=0.25)
        annotated = draw_detections(image, detections, class_names=['leaf'])
        ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        ax.set_title(f'{img_path.name} — {len(detections)} leaves (leaf_v3_ft)', fontsize=10)
        ax.axis('off')
    fig.tight_layout()
    plt.show()
    print('Too few real photos for a numeric held-out metric — eyeball the boxes above. '
          'Add more real photos, re-run C13-C15 once you have 10+, for a real measured number.')